# 03 - Minimal DORA Evidence Tree

This notebook is the shortest rigorous path from definitions to an evidence tree:

1. define a small schema
2. write source-backed regulatory anchors
3. define one rule and one derivation
4. evaluate a candidate review finding
5. display the evidence tree that explains the candidate

The input facts are not invented company facts. They are regulatory anchors from official EUR-Lex pages:

- Regulation (EU) 2022/2554, Article 5(1) and Article 8(1): https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32022R2554
- Delegated Regulation (EU) 2024/1774, Article 5(1): https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1774

The derived finding is deliberately narrow: these source-backed DORA ICT risk-management anchors are present in the review set. It is not a compliance decision about any financial entity.

**Prerequisites:** none beyond the local `kernel` package. No external engine, network call, or LLM call is required.  
**Next:** [04_ecss_souffle_compliance.ipynb](04_ecss_souffle_compliance.ipynb) for a larger compliance walkthrough, or [07_evidence_graph_multi_engine.ipynb](07_evidence_graph_multi_engine.ipynb) for the full multi-engine explain demo.

## 0. Setup

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))

In [ ]:
from kernel.sdk import SDKStore, Entity, Identity, Field, Rule, RuleRef, Derivation, vars
from kernel.core.store._candidate_evidence_tree import build_candidate_evidence_tree
from kernel.core.store._candidate_evidence_tree_summary import summarize_candidate_evidence_tree_dict
from kernel.core.store._candidate_evidence_tree_narrative import render_candidate_evidence_tree_narrative
from service.static_ui import render_candidate_evidence_html

try:
    from IPython.display import HTML, display
except Exception:  # Plain Python fallback.
    HTML = None
    display = None

## 1. Define The Review Vocabulary

`RegulatoryAnchor` records a specific source location and the obligation category we will use in the rule. `ReviewCase` is the review artifact that will receive the derived finding.

In [ ]:
DORA_URL = "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32022R2554"
RTS_URL = "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1774"
REVIEW_CASE_ID = "dora-minimal-ict-risk-chain"


class RegulatoryAnchor(Entity):
    anchor_id: str = Identity(primary_key=True)
    instrument: str = Field(cardinality="single")
    article: str = Field(cardinality="single")
    obligation: str = Field(cardinality="single")
    source_url: str = Field(cardinality="single")


class ReviewCase(Entity):
    case_id: str = Identity(primary_key=True)
    scope: str = Field(cardinality="single")
    finding: str = Field(cardinality="multi")


classes = [RegulatoryAnchor, ReviewCase]
sdk = SDKStore.from_schema_classes(classes, default_row_format="dict")
print("schema predicates:", len(sdk.schema_ir["predicates"]))

## 2. Write Source-Backed Facts

The regulatory facts are article identifiers, obligation labels, and source URLs. The source text is intentionally not copied into the notebook; the article id plus EUR-Lex URL is the audit handle.

In [ ]:
SOURCE_ANCHORS = [
    {
        "anchor_id": "DORA-2022-2554-ART-5-1",
        "instrument": "Regulation (EU) 2022/2554",
        "article": "Article 5(1)",
        "obligation": "ict_governance_framework",
        "source_url": DORA_URL,
    },
    {
        "anchor_id": "DORA-2022-2554-ART-8-1",
        "instrument": "Regulation (EU) 2022/2554",
        "article": "Article 8(1)",
        "obligation": "ict_asset_identification_and_documentation",
        "source_url": DORA_URL,
    },
    {
        "anchor_id": "DORA-2024-1774-ART-5-1",
        "instrument": "Delegated Regulation (EU) 2024/1774",
        "article": "Article 5(1)",
        "obligation": "ict_asset_management_procedure",
        "source_url": RTS_URL,
    },
]

case_tx = sdk.batch(meta={"source": "review-scope", "trace_id": "dora-review-case"})
case = case_tx.entity(ReviewCase, case_id=REVIEW_CASE_ID)
case.scope.set("DORA Chapter II plus RTS Article 5 minimal source-chain review")
case_tx.commit()

source_tx = sdk.batch(meta={"source": "EUR-Lex", "trace_id": "dora-source-anchors"})
for row in SOURCE_ANCHORS:
    anchor = source_tx.entity(RegulatoryAnchor, anchor_id=row["anchor_id"])
    anchor.instrument.set(row["instrument"])
    anchor.article.set(row["article"])
    anchor.obligation.set(row["obligation"])
    anchor.source_url.set(row["source_url"])
source_tx.commit()

REF_LABELS = {sdk.ref(ReviewCase, case_id=REVIEW_CASE_ID): REVIEW_CASE_ID}
for row in SOURCE_ANCHORS:
    REF_LABELS[sdk.ref(RegulatoryAnchor, anchor_id=row["anchor_id"])] = row["anchor_id"]

pprint({"review_case": REVIEW_CASE_ID, "source_anchor_count": len(SOURCE_ANCHORS)})

## 3. Define The Rule And Derivation

The rule checks that the review set contains three source-backed anchors:

- DORA governance/control framework anchor
- DORA identification/documentation anchor
- DORA RTS asset-management procedure anchor

The derivation turns that rule result into one review finding. The `RuleRef` is important because it gives the evidence tree a visible rule-chain branch instead of a flat list of facts.

In [ ]:
with vars("case", "scope", "g", "i", "p", "out") as (case, scope, g, i, p, out):
    source_chain_rule = Rule(
        id="q.dora_minimal_source_chain",
        version="1.0.0",
        select=[g, i, p],
        where=[
            RegulatoryAnchor(g),
            g.instrument == "Regulation (EU) 2022/2554",
            g.article == "Article 5(1)",
            g.obligation == "ict_governance_framework",
            g.source_url == DORA_URL,
            RegulatoryAnchor(i),
            i.instrument == "Regulation (EU) 2022/2554",
            i.article == "Article 8(1)",
            i.obligation == "ict_asset_identification_and_documentation",
            i.source_url == DORA_URL,
            RegulatoryAnchor(p),
            p.instrument == "Delegated Regulation (EU) 2024/1774",
            p.article == "Article 5(1)",
            p.obligation == "ict_asset_management_procedure",
            p.source_url == RTS_URL,
        ],
        expose=True,
    )

    conclusion_drv = Derivation(
        id="drv.dora_minimal_review_conclusion",
        version="1.0.0",
        where=[
            ReviewCase(case),
            case.scope == scope,
            RuleRef(source_chain_rule)(g, i, p),
            out == "source-backed DORA ICT risk-management anchors are present",
        ],
        head=ReviewCase.finding(finding=out),
    )

print("rule:", source_chain_rule.id, source_chain_rule.version)
print("derivation:", conclusion_drv.id, conclusion_drv.version)

## 4. Evaluate A Candidate Finding

`evaluate` does not write the derived finding. It creates a candidate plus support artifacts. That is exactly where evidence trees are useful: they let a reviewer inspect why a result exists before accepting it.

In [ ]:
candidates = sdk.evaluate(conclusion_drv, mode="native")
if len(candidates) != 1:
    raise AssertionError(f"expected exactly one candidate, got {len(candidates)}")

candidate = candidates[0]
support_digest = sdk.store.get_candidate_support_digest(candidate.candidate_id)
support_kind = sdk.store.get_candidate_support_kind(candidate.candidate_id)

pprint(
    {
        "candidate_id": candidate.candidate_id,
        "target": candidate.target,
        "support_kind": support_kind,
        "support_digest": support_digest,
        "derived_value": candidate.payload["terms"][1]["value"],
    }
)

## 5. Build The Evidence Tree

The tree starts from the candidate id. From there it follows the support digest to the direct derivation support, then follows the `RuleRef` edge into the referenced rule support, and finally lands on the source-backed assertion leaves.

In [ ]:
def assertion_detail(asrt_id):
    claim = sdk.ledger.get_claim(asrt_id)
    if claim is None:
        return None
    claim_args = [
        {"idx": row.idx, "val": "" if row.val_atom is None else str(row.val_atom), "tag": row.tag}
        for row in sdk.ledger.find_claim_args(asrt_id=asrt_id)
    ]
    claim_args.sort(key=lambda row: (row["idx"], row["tag"], row["val"]))

    flat_meta = {}
    for key in ("source", "trace_id"):
        rows = sdk.ledger.find_meta(asrt_id=asrt_id, key=key, kind="str")
        if rows:
            flat_meta[key] = str(rows[0].value)

    detail = {
        "asrt_id": asrt_id,
        "claim": {"asrt_id": asrt_id, "pred_id": claim.pred_id, "e_ref": claim.e_ref},
        "claim_args": claim_args,
    }
    if flat_meta:
        detail["flat_meta"] = flat_meta
    return detail


support = sdk.store.explain_support(support_digest)
tree = build_candidate_evidence_tree(
    candidate_id=candidate.candidate_id,
    support_digest=support_digest,
    support_kind=support_kind,
    support=support,
    assertion_lookup=assertion_detail,
    support_lookup=sdk.store.explain_support,
)

In [ ]:
def label(value):
    if value in REF_LABELS:
        return REF_LABELS[value]
    if value == DORA_URL:
        return "EUR-Lex CELEX:32022R2554"
    if value == RTS_URL:
        return "EUR-Lex CELEX:32024R1774"
    return str(value)


def claim_values(node):
    return [label(arg.get("val")) for arg in node.get("claim_args", [])]


def print_tree(node, indent=0):
    prefix = "  " * indent
    kind = node.get("node_kind", "?")

    if kind == "candidate_result":
        binding = node.get("binding", {})
        print(f"{prefix}- CANDIDATE: {label(binding.get('$out', '<derived finding>'))}")
    elif kind == "support_section":
        print(f"{prefix}- SUPPORT: direct facts and checks used by this step")
    elif kind == "rule_ref_section":
        print(f"{prefix}- RULE CHAIN: referenced rules that need their own support")
    elif kind == "rule_ref":
        print(f"{prefix}- RULE REF: {node.get('rule_ref_id')} v{node.get('rule_ref_version')}")
    elif kind == "referenced_support":
        print(f"{prefix}- REFERENCED SUPPORT: proof returned by the rule")
    elif kind == "predicate_witness_group":
        print(f"{prefix}- NEEDS FACT: {node.get('pred_id')} ({node.get('assertion_count')} assertion)")
    elif kind == "assertion_fact":
        meta = node.get("fact_meta", {})
        source = f" source={meta.get('source')}" if meta.get("source") else ""
        print(f"{prefix}- FACT: {node.get('pred_id')} -> {claim_values(node)}{source}")
    elif kind == "non_fact_check":
        print(f"{prefix}- CHECK: {node.get('check_kind')} is {node.get('status')}")
    else:
        print(f"{prefix}- {kind}: {node.get('title', '')}")

    for child in node.get("children", []):
        print_tree(child, indent + 1)


print_tree(tree["root"])

## 6. Read The Tree As A Reviewer

The deterministic summary is useful in a meeting because it compresses the raw tree into review signals: how many witness assertions exist, whether the tree had to recurse through a rule, and whether any support was unresolved.

In [ ]:
summary = summarize_candidate_evidence_tree_dict(tree)
narrative = render_candidate_evidence_tree_narrative(summary, tree=tree)

pprint(
    {
        "witness_assertions": summary["witness_assertion_count"],
        "rule_refs": summary["rule_ref_count"],
        "recursive_depth": summary["recursive_depth"],
        "has_unresolved_support": summary["has_unresolved"],
    }
)
print("headline:", narrative["headline"])
print("why it matters:")
print("- candidate_result is the conclusion under review")
print("- rule_ref shows which rule produced part of the support")
print("- assertion_fact leaves show the source-backed facts used by the rule")
print("- unresolved_support would make gaps visible instead of hiding them")

## 7. Optional Visual Evidence Page

The same raw tree can be rendered as HTML for a review meeting. This is a local render of the existing static evidence UI, not a separate proof system.

In [ ]:
html = render_candidate_evidence_html(tree, narrative=narrative)
if display is not None and HTML is not None:
    display(HTML(html))
else:
    print(html[:1000])

## 8. Accept After Review

Acceptance is the audited materialization step. The important point is that the evidence tree was available before this write, so a reviewer can inspect the support before accepting the candidate.

In [ ]:
accept_result = sdk.accept(candidate, approved_by="meeting-review", note="accepted after evidence-tree review")
print("accepted_count:", accept_result.accepted_count)
print("written_assertions:", accept_result.written_assertions)

review_case = sdk.get(ReviewCase, case_id=REVIEW_CASE_ID)
print("materialized finding:", review_case.finding)